# Stage1 分钟级配置动作空间预检

逐分钟 Qlib 来源，逐层点时 f696+b472 开盘母本。只核对时钟、源完整性及当前现金/风险容量；不读取未来收益、不拟合、不生成交易策略。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
import hashlib, json
from pathlib import Path
import exchange_calendars as xc
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.backtest.event_guard_clock import guard_slot
from src.workflow.minute_action_value import _source

root=Path.cwd(); card=root/"research/stage1_intraday_allocation_action_space_20261028"
raw=(card/"plan.json").read_bytes(); plan=json.loads(raw)
parent_plan_path=root/plan["parent_plan"]
parent=json.loads(parent_plan_path.read_text())
assert hashlib.sha256((root/plan["provider_uri"]/"qlib_store_manifest.json").read_bytes()).hexdigest()==plan["provider_manifest_sha256"]
assert parent["provider_manifest_sha256"]==plan["provider_manifest_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
source,receipt=_source(root,parent["training_source"],["analysis/native/dated_f696_0"])
frame_source,frame_receipt=_source(root,parent["frame_source"],["inputs/covered_frame"])
book=source["analysis/native/dated_f696_0"]
frame=frame_source["inputs/covered_frame"]
assert len(frame)==plan["expected_parent_sessions"] and frame.trade_date.equals(book["authority"].trade_date)
bars=D.features(["SOXL","SOXX"],["$close","$price_valid"],start_time=parent["training_start"]+" 09:31:00",end_time=parent["training_end"]+" 16:00:00",freq="1min")
wide=[]
for symbol in ("SOXL","SOXX"):
    part=bars.xs(symbol,level="instrument").copy()
    part.columns=[symbol+"_"+column.removeprefix("$") for column in part]
    wide.append(part)
quotes=pd.concat(wide,axis=1).sort_index()
schedule=xc.get_calendar("XNYS",start=frame.trade_date.min(),end=frame.trade_date.max()).schedule
closes={pd.Timestamp(day).tz_localize(None):pd.Timestamp(row.close).tz_convert("America/New_York").tz_localize(None) for day,row in schedule.iterrows()}
rows=[]
for index,row in frame.reset_index(drop=True).iterrows():
    day=pd.Timestamp(row.trade_date);actual=book["authority"].iloc[index]
    status="eligible";event_rows=0;cash_room=0;rotation_room=0;room_var=False;first=None;last=None
    if index+1==len(frame):status="terminal"
    elif bool(actual.recovery_action):status="recovery"
    asset=quotes.loc[quotes.index.normalize()==day]
    if asset.empty or not asset[["SOXL_price_valid","SOXX_price_valid"]].eq(1).all().all():status="source_invalid"
    if status=="eligible":
        opened=day+pd.Timedelta(hours=9,minutes=47);close=closes[day]
        valid=(asset.index>=opened+pd.Timedelta(minutes=1)) & (asset.index+pd.Timedelta(minutes=2)<close)
        event=asset.loc[valid]
        if event.empty:status="no_event"
        else:
            pos=book["native_positions"].get(opened)
            assert pos is not None
            q=np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ("SOXL","SOXX")],float)
            cash=float(pos.get_cash());price=event[["SOXL_close","SOXX_close"]].to_numpy(float)
            nav=price@q+cash
            assert np.isfinite(nav).all() and (nav>0).all()
            weights=price*q/nav[:,None]
            beta=3*weights[:,0]+weights[:,1]
            cash_weight=cash/nav
            cap=float(actual.current_beta_cap)
            room=np.minimum(cash_weight,np.maximum(0,cap-beta))
            event_rows=len(event);cash_room=int((room>1e-10).sum())
            rotation_room=int(((weights[:,0]>1e-10)&(cash_weight>1e-10)).sum())
            room_var=bool(np.ptp(room)>1e-10)
            first=event.index[0]+pd.Timedelta(minutes=1);last=event.index[-1]+pd.Timedelta(minutes=1)
            assert guard_slot(event.index[0],session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=close,original_open_fill=opened).decision_at==first
            assert guard_slot(event.index[-1],session_open=day+pd.Timedelta(hours=9,minutes=30),session_close=close,original_open_fill=opened).decision_at==last
    rows.append({"trade_date":day,"year":day.year,"status":status,"event_minutes":event_rows,"cash_to_risk_target_room_minutes":cash_room,"beta_neutral_rotation_target_room_minutes":rotation_room,"cash_to_risk_room_varies_intraday":room_var,"first_decision_at":str(first) if first is not None else None,"last_decision_at":str(last) if last is not None else None,"half_day":bool(closes[day].hour<16)})
perday=pd.DataFrame(rows)
annual=[]
for year,part in perday.groupby("year"):
    annual.append({"year":int(year),"sessions":len(part),"eligible_days":int(part.status.eq("eligible").sum()),"event_minutes":int(part.event_minutes.sum()),"days_with_cash_to_risk_target_room":int(part.cash_to_risk_target_room_minutes.gt(0).sum()),"days_with_beta_neutral_rotation_target_room":int(part.beta_neutral_rotation_target_room_minutes.gt(0).sum()),"days_with_intraday_cash_room_variation":int(part.cash_to_risk_room_varies_intraday.sum()),"half_days":int(part.half_day.sum())})
audit={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(raw).hexdigest(),"parent_recorder_id":parent["training_source"]["recorder_id"],"frame_recorder_id":parent["frame_source"]["recorder_id"],"provider_manifest_sha256":plan["provider_manifest_sha256"],"native_qlib_source":True,"source_receipts":{"parent":len(receipt),"frame":len(frame_receipt)},"sessions":len(perday),"status_counts":perday.status.value_counts().to_dict(),"annual":annual,"event_minutes":int(perday.event_minutes.sum()),"first_decision_at":perday.first_decision_at.dropna().min(),"last_decision_at":perday.last_decision_at.dropna().max(),"new_model_fits":0,"new_native_accounts":0,"future_outcomes_read":0,"historical_research_passed":False,"decision":"intraday_clock_target_space_diagnostic_only"}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"status_counts":audit["status_counts"],"event_minutes":audit["event_minutes"],"annual":annual},ensure_ascii=False,indent=2))

{
  "status_counts": {
    "eligible": 1044,
    "recovery": 10,
    "source_invalid": 1,
    "terminal": 1
  },
  "event_minutes": 384660,
  "annual": [
    {
      "year": 2018,
      "sessions": 53,
      "eligible_days": 53,
      "event_minutes": 19250,
      "days_with_cash_to_risk_target_room": 50,
      "days_with_beta_neutral_rotation_target_room": 53,
      "days_with_intraday_cash_room_variation": 50,
      "half_days": 2
    },
    {
      "year": 2019,
      "sessions": 252,
      "eligible_days": 252,
      "event_minutes": 92700,
      "days_with_cash_to_risk_target_room": 222,
      "days_with_beta_neutral_rotation_target_room": 252,
      "days_with_intraday_cash_room_variation": 222,
      "half_days": 3
    },
    {
      "year": 2020,
      "sessions": 253,
      "eligible_days": 242,
      "event_minutes": 89180,
      "days_with_cash_to_risk_target_room": 201,
      "days_with_beta_neutral_rotation_target_room": 242,
      "days_with_intraday_cash_room_variation":

## 中文结论

Qlib 一分钟来源和逐层点时 f696+b472 原账户核对后，1056 日中有 1044 日、384660 个有效盘中决策点，最早 09:49、最晚 15:58。现金与原 beta 上限共同形成正目标空间的 929 天，该空间随当日分钟价格变化。10 个恢复日、1 个异常源日、1 个终端日保留排除。来源、年度和半日市见 `audit.json`。

决定 `intraday_clock_target_space_diagnostic_only`。本卡零未来收益、零模型、零新账户；正目标空间不代表整股成交或 SOXX 超额。下一步按 v4 先冻结动作幅度、整股费用及每分钟与 HOLD/等待的同状态局部标签，再检验分钟信息是否改变成交时刻和完整日历净价值。历史分钟首次接收仍是 `needs_data`。